# 🔦 OPERAÇÃO FAROL · FASE 2
### Simulado Tabular 1 · Trilha de Dados Tabulares · rede neural sobre os dados da Fase 1

---

> **"Vocês me entregaram uma floresta que acertava bem. Agora me dizem que uma rede neural
> pode fazer melhor. Pode mesmo? E quanto custa descobrir?"**
>
> — Marina Alencar, coordenação pedagógica

## 📨 A Marina voltou

> A tabela é a mesma da Fase 1: um estudante por linha, com notas e taxas de aprovação dos
> dois semestres, situação financeira, família e curso. O risco de evasão continua em três
> níveis: **baixo**, **médio** e **alto**.
>
> Uma coisa mudou na tabela, e eu fiz de propósito. Na Fase 1, o curso vinha espalhado em
> dezessete colunas de 0 e 1. Agora ele volta a ser **uma coluna só**, com o nome do curso,
> como chega da secretaria. Ocupação dos pais e qualificação anterior também são
> **categorias**: o número é um código, não uma quantidade. Vocês vão decidir como a rede
> lê isso.
>
> — Marina Alencar

---

*Mesmos dados oficiais da Operação Farol. O teste oficial não tem gabarito público; por
isso 20% das linhas rotuladas viraram o teste oculto deste simulado.*

## 🎯 A sua missão


Treinar uma **rede neural** que aponte o risco de evasão de cada estudante, comparando três
arquiteturas — **rede densa**, **rede com embeddings** e **TabNet** — e justificando a
escolha com números: F1-macro, forma das curvas e **custo de treino**.


### O que muda da Fase 1 para cá

| Na Fase 1 | Na Fase 2 |
|:---|:---|
| Árvore, floresta, regressão: `.fit` e pronto | **Redes neurais**: um laço de épocas e lotes, com hiperparâmetros |
| Trocar de modelo era trocar uma palavra | Cada modelo quer **a entrada de um jeito** (o preparo virou metade da escolha) |
| Um número no fim | **Duas curvas** (treino e validação) contando o filme inteiro |

### 🗺️ O mapa da operação

| Bloco | O que vocês fazem | Como |
|:---|:---|:---|
| **1 · EXPLORAR** | Medir o piso, o sinal das colunas e as cardinalidades | ✏️ **2 lacunas** |
| **2 · CONFIGURAR** | Escolher o modelo e girar os botões | Só arrastando |
| **3 · MODELAR** | Células A (preparo), B (treino) e C (avaliação) | ✏️ **11 lacunas** + Zona Livre |
| **4 · AVALIAR** | Ler as curvas, a matriz, o custo e os grupos | Só clicando |

> ⚠️ **Antes de começar.** Esta trilha roda bem em **CPU**: a tabela é pequena e as redes
> também. Se quiserem, liguem a GPU (`Ambiente de execução` → `Alterar o tipo`), mas não é
> obrigatório. Usem `Arquivo` → `Salvar uma cópia no Drive` antes de começar.
>
> 📎 **Simulado de preparação.** Formato dos cadernos da Fase 2, com dados públicos. O
> gabarito do teste fica com o corretor (`simulados/corrigir.py`), nunca dentro do caderno.

In [ ]:
#@title 🛠️ Preparação: abrindo os registros { display-mode: "form" }
# Os arquivos treino.csv e teste.csv ficam na mesma pasta deste caderno.
URL_TREINO = ""
URL_TESTE = ""
SEMENTE = 42

import pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, recall_score, confusion_matrix,
                             classification_report)

ALVO = "risco_evasao"
NOMES = {0: 'Baixo', 1: 'Médio', 2: 'Alto'}
CORES = ["#3DCB55", "#FFC107", "#E6177A"]
# Colunas de categoria: o número (ou o texto) delas é um rótulo, não uma quantidade.
CATEGORICAS = ['curso', 'ocupacao_mae', 'ocupacao_pai', 'qualificacao_anterior']

def _carregar(url, nome_local):
    """Usa a cópia ao lado do caderno; senão tenta o endereço; senão pede o envio."""
    if pathlib.Path(nome_local).exists():
        return pd.read_csv(nome_local)
    if url:
        print(f"Baixando {nome_local}...")
        try:
            return pd.read_csv(url)
        except Exception as erro:
            print(f"  não deu certo ({erro.__class__.__name__}).")
    from google.colab import files
    print(f"  envie o arquivo {nome_local}:")
    files.upload()
    return pd.read_csv(nome_local)

dados = _carregar(URL_TREINO, "treino.csv")
_teste = _carregar(URL_TESTE, "teste.csv")
IDS_TESTE = _teste["id_estudante"].to_numpy()
dados_teste = _teste.drop(columns=["id_estudante"])
N_TESTE = len(dados_teste)
NUMERICAS = [c for c in dados.columns if c not in CATEGORICAS + [ALVO]]

print("Ambiente carregado.\n")
print(f"  Conjunto público (com respostas) .... {len(dados):>5} registros")
print(f"  Conjunto de teste (sem respostas) ... {N_TESTE:>5} registros  [tamanho fixo]")
print(f"  Colunas numéricas ................... {len(NUMERICAS):>5}")
print(f"  Colunas de categoria ................ {len(CATEGORICAS):>5}  {CATEGORICAS}\n")
for k, n in NOMES.items():
    q = (dados[ALVO] == k).sum()
    print(f"     {n:<11} ({k}) .................. {q:>5}   {q/len(dados)*100:4.1f}%")
display(dados.head())

---
# 🔍 BLOCO 1 · EXPLORAR
## *Três números para anotar antes de treinar qualquer coisa*

> No caderno do desafio, **nada disto é calculado para vocês**. Os três números voltam no
> Bloco 2 e no Bloco 3.

1. **O piso** — o F1-macro do *chute preguiçoso*, um modelo que responde sempre a classe
   mais comum. É contra ele que se mede se a rede aprendeu alguma coisa.
2. **As colunas de maior sinal** — as primeiras a conferir quando o resultado vier estranho.
   E se uma coluna sozinha explicar quase tudo, isso quase nunca é sorte: é **vazamento**.
3. **Quantos valores tem cada categoria** — obrigatório. Sem esse número, dois dos três
   modelos nem rodam.

In [ ]:
# ✏️ SUA VEZ · Os três números do Bloco 1
# Substituam cada ____ pelo que falta.
y_publico = dados[ALVO].to_numpy()

# 01 · O PISO: o chute preguiçoso responde sempre a classe mais comum.
classe_mais_comum = pd.Series(y_publico).mode()[0]
chute = np.full_like(y_publico, classe_mais_comum)
# LACUNA 1 ▸ que média do F1 é a métrica da fase? (a que dá o mesmo peso a cada classe)
piso = f1_score(y_publico, chute, average=____)

# 03 · QUANTOS VALORES TEM CADA CATEGORIA
# LACUNA 2 ▸ qual método do pandas conta quantos valores DIFERENTES uma coluna tem?
cardinalidades = {c: int(dados[c].____()) for c in CATEGORICAS}

acc_chute = accuracy_score(y_publico, chute)
print(f"O PISO · o chute preguiçoso ('{NOMES[classe_mais_comum]}' para todos)")
print(f"   acurácia ..... {acc_chute*100:5.1f}%   ← parece que aprendeu")
print(f"   F1-macro ..... {piso*100:5.1f}%   ← a métrica da fase não se deixa enganar\n")
print("QUANTOS VALORES TEM CADA CATEGORIA")
for c, k in cardinalidades.items():
    print(f"   {c:<24} {k:>3} valores")

In [ ]:
#@title 📡 02 · O medidor de sinal: quanto cada coluna, sozinha, diz sobre a resposta { display-mode: "form" }
from sklearn.feature_selection import mutual_info_classif
_X = dados.drop(columns=[ALVO]).copy()
for _c in CATEGORICAS:
    _X[_c] = _X[_c].astype("category").cat.codes
_sinal = pd.Series(
    mutual_info_classif(_X, y_publico, discrete_features=[c in CATEGORICAS for c in _X.columns],
                        random_state=SEMENTE),
    index=_X.columns).sort_values()
# Normalizado pela entropia da resposta: 1 = a coluna sozinha determina a classe.
_p = np.bincount(y_publico) / len(y_publico)
_sinal = (_sinal / -(_p[_p > 0] * np.log(_p[_p > 0])).sum()).clip(0, 1)

fig, ax = plt.subplots(figsize=(8.5, max(4, len(_sinal) * 0.27)))
ax.barh(_sinal.index, _sinal.values,
        color=["#E6177A" if v > .5 else "#00B8D4" for v in _sinal.values])
ax.axvline(.15, color="#999", ls="--", lw=1)
ax.set_xlim(0, 1); ax.set_xlabel("sinal (0 = nenhum · 1 = a coluna resolve sozinha)")
ax.set_title("Medidor de sinal (informação mútua, normalizada)", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

top3 = list(_sinal.sort_values(ascending=False).index[:3])
print("AS TRÊS COLUNAS DE MAIOR SINAL:", ", ".join(top3))
if _sinal.max() > .5:
    print(f"\n[!] '{_sinal.idxmax()}' sozinha explica quase tudo. Isso quase nunca é sorte:")
    print("    investiguem se é informação que só existe DEPOIS de saber a resposta.")
elif _sinal.max() < .05:
    print("\n[!] Nenhuma coluna passa de 0,05. Nenhum modelo vai longe aqui, por mais caro.")
else:
    print("\nO sinal está espalhado: é a combinação das colunas que classifica.")

In [ ]:
#@title 🔗 Duas colunas que contam a mesma coisa { display-mode: "form" }
_corr = dados[NUMERICAS].corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(_corr.abs(), cmap="Purples", vmin=0, vmax=1)
ax.set_xticks(range(len(NUMERICAS))); ax.set_xticklabels(NUMERICAS, rotation=90, fontsize=7)
ax.set_yticks(range(len(NUMERICAS))); ax.set_yticklabels(NUMERICAS, fontsize=7)
plt.colorbar(im, fraction=.04); ax.set_title("|correlação| entre colunas numéricas", fontweight="bold")
plt.tight_layout(); plt.show()
_pares = [(a, b, _corr.loc[a, b]) for i, a in enumerate(NUMERICAS) for b in NUMERICAS[i+1:]
          if abs(_corr.loc[a, b]) > .7]
if _pares:
    print("Pares que andam juntos (|r| > 0,7):")
    for a, b, r in sorted(_pares, key=lambda t: -abs(t[2])):
        print(f"   {a}  ×  {b}   r = {r:+.2f}")
    print("\nInformação repetida não soma. E divide a importância entre as duas na hora de")
    print("interpretar: nenhuma das duas parece importante, embora a informação seja.")
else:
    print("Nenhum par passa de |r| = 0,7.")

In [ ]:
#@title 📊 Investigue: escolha a variável E o gráfico { display-mode: "form" }
variavel = "taxa_aprovacao_2sem"  #@param ["taxa_aprovacao_2sem", "taxa_aprovacao_1sem", "uc_2sem_nota_media", "uc_1sem_nota_media", "idade_na_matricula", "mensalidades_em_dia", "devedor", "bolsista", "curso", "ocupacao_mae", "qualificacao_anterior", "turno"]
tipo_de_grafico = "Boxplot"  #@param ["Boxplot", "Histograma sobreposto", "Barras de proporcao"]

serie = dados[variavel]
fig, ax = plt.subplots(figsize=(8.5, 4.2))
if tipo_de_grafico == "Barras de proporcao" or variavel in CATEGORICAS:
    tab = pd.crosstab(serie, dados[ALVO], normalize="index")[list(NOMES)]
    tab.columns = list(NOMES.values())
    tab.plot(kind="bar", stacked=True, color=CORES, ax=ax, edgecolor="white")
    ax.set_ylabel("proporção dentro de cada valor"); ax.legend(bbox_to_anchor=(1, 1))
elif tipo_de_grafico == "Boxplot":
    bp = ax.boxplot([serie[dados[ALVO] == k] for k in NOMES], patch_artist=True,
                    tick_labels=list(NOMES.values()), medianprops=dict(color="black", linewidth=2))
    for p, cor in zip(bp["boxes"], CORES): p.set_facecolor(cor); p.set_alpha(.8)
else:
    bins = np.histogram_bin_edges(serie, bins=25)
    for k, cor in zip(NOMES, CORES):
        ax.hist(serie[dados[ALVO] == k], bins=bins, alpha=.55, color=cor, density=True,
                label=NOMES[k], edgecolor="white")
    ax.legend()
ax.set_title(f"'{variavel}' por classe", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
if variavel in CATEGORICAS:
    print("Coluna de CATEGORIA: o número/texto é um rótulo. Na rede densa ela vira colunas")
    print("de 0 e 1; nas outras duas, um índice com tabela própria.")

---
# ⚙️ BLOCO 2 · CONFIGURAR
## *O preparo deixou de ser detalhe e virou metade da escolha*

| Conjunto | Tamanho | Vocês têm as respostas? | Para que serve |
|:---|:---|:---|:---|
| **Público** | 2.588 estudantes | Sim | Treinar e validar |
| **Teste** | **648 (fixo)** | Não | Comparação final (`simulados/corrigir.py`) |

### 📗 As três fichas

| | 🧱 Rede densa | 🧩 Rede com embeddings | 🌳 TabNet |
|:---|:---|:---|:---|
| **O que é** | Camadas de neurônios empilhadas, sem nenhuma ideia além disso | A mesma rede, mas cada valor de categoria vira um punhado de números que o treino ajusta | Arquitetura feita para tabela: decide em vários passos quais colunas olhar |
| **Categorias entram como** | colunas de 0 e 1 (`pd.get_dummies`) | índices inteiros + a lista de **cardinalidades** | índices inteiros + **duas listas**: posições (`cat_idxs`) e cardinalidades (`cat_dims`) |
| **Régua (`StandardScaler`)** | **obrigatória**, em tudo | só nas colunas **numéricas** | **dispensável**: normaliza por dentro |
| **Taxa de aprendizado** | `1e-3` | `1e-3` | `2e-2` (vinte vezes maior: foi feito para começar rápido e ir freando) |
| **Botões próprios** | largura, dropout | + tamanho do embedding (8 costuma bastar) | `n_steps`, `n_d`/`n_a`, lote virtual |
| **A pegadinha** | Sem régua nas numéricas, trava | Cardinalidade errada quebra na hora de prever | O lote virtual precisa **dividir exatamente** o lote; trocar posições por cardinalidades dá índice fora da tabela |
| **O que esperar** | Rápida e mais competitiva do que parece | Costuma ganhar quando há categoria com muitos valores | Mais lenta; é a única que diz quais colunas usou |

### 🎛️ As quatro peças que já vêm prontas

| Peça | O que faz | Sem ela |
|:---|:---|:---|
| **Otimizador** `AdamW` | Move cada peso depois que o erro apontou a direção, com memória dos passos anteriores | `SGD` simples: desce mais devagar e com mais ruído |
| **Agendador** `CosineAnnealingLR` | O passo vai encurtando ao longo do treino | Passo grande até o fim; a curva fica tremida perto do fundo |
| **Parada antecipada** | Para quando a validação não melhora por `paciencia` épocas | Treina até o fim, piorando |
| **Ponto de verificação** | Guarda a cópia da **melhor** época | A parada entrega o modelo da última época, já piorando |

### Como ler os botões

- **Taxa de aprendizado** — o tamanho do passo. Baixa demais: as curvas descem lisas e
  **devagar** e a melhor época é a última. Alta demais: a curva **serrilha e sobe**. São
  dois problemas com dois formatos: confundi-los faz mexer o botão na direção errada.
- **Largura e camadas** — capacidade. Rede estreita demais: as duas curvas param **altas e
  juntas** (subajuste). Larga demais sem freio: treino desce, validação sobe (sobreajuste).
- **Dropout** — o freio. Desliga neurônios ao acaso **só no treino**. Comecem em 0,30.
- **Épocas e paciência** — quantas voltas cabem, e quantas sem melhora o treino tolera.

> Os valores iniciais abaixo são **deliberadamente modestos**.

In [ ]:
#@title 🎛️ Parâmetros do experimento { display-mode: "form" }
modelo_escolhido = "Rede densa"  #@param ["Rede densa", "Rede com embeddings", "TabNet"]
tamanho_da_validacao = 20  #@param {type:"slider", min:10, max:40, step:5}
taxa_de_aprendizado = "1e-4"  #@param ["1e-5", "1e-4", "1e-3", "1e-2", "2e-2"]
epocas = 40  #@param {type:"slider", min:5, max:200, step:5}
paciencia = 10  #@param {type:"slider", min:2, max:30, step:1}
lote = 256  #@param [64, 128, 256, 512, 1024] {type:"raw"}
largura = "64"  #@param ["16", "64", "128, 64", "256, 128", "512, 256, 128", "1024, 1024, 1024"]
dropout = 0.0  #@param {type:"slider", min:0, max:0.6, step:0.05}
dim_embedding = 8  #@param {type:"slider", min:2, max:24, step:1}
n_steps = 3  #@param {type:"slider", min:3, max:8, step:1}
n_d = 8  #@param [8, 16, 24, 32] {type:"raw"}

taxa = float(taxa_de_aprendizado)
camadas = tuple(int(w) for w in largura.split(","))
print("PARÂMETROS DEFINIDOS")
print(f"   Modelo ............. {modelo_escolhido}")
print(f"   Validação .......... {tamanho_da_validacao}%")
print(f"   Taxa ............... {taxa:g}")
print(f"   Épocas / paciência . {epocas} / {paciencia}   |   lote {lote}")
if modelo_escolhido == "TabNet":
    print(f"   n_steps / n_d ...... {n_steps} / {n_d}")
    if taxa < 1e-2:
        print("\n[!] A ficha do TabNet pede 2e-2. Com a taxa da rede densa, o treino fica")
        print("    inacabado dentro do limite de épocas.")
else:
    print(f"   Camadas ............ {camadas}  |  dropout {dropout}")
    if modelo_escolhido == "Rede com embeddings":
        print(f"   Embedding .......... {dim_embedding} números por valor de categoria")
    if taxa >= 1e-2:
        print("\n[!] Taxa alta para uma rede treinada do zero (a ficha pede 1e-3).")

---
# ✏️ BLOCO 3 · MODELAR
## *Três células, sempre as mesmas*

| Célula | O que faz | O que vocês escrevem |
|:---|:---|:---|
| **A · preparo** | Separa treino e validação e transforma a entrada. **É a única que muda de verdade entre os três modelos** | lacunas do preparo |
| **B · treino** | Constrói o modelo, configura as quatro peças e treina | lacunas de hiperparâmetro + **Zona Livre** |
| **C · avaliação** | Calcula a métrica e registra o experimento | nada: **vem pronta, é para ler** |

### O erro que não dá erro: a régua antes da separação

Se o `StandardScaler` aprender a média e o desvio com a tabela **inteira** e só depois a
tabela for separada, a régua leva para o treino uma informação que veio da validação. O
código roda, nenhuma mensagem aparece, e a nota de validação **sobe** — falsamente. Na
plataforma, ela cai. A verificação deste bloco confere exatamente isso.

### Quando o TabNet quebra com uma mensagem estranha

`running_mean should contain 22 elements not 23`, `index out of range in self`: essas
mensagens quase nunca falam do que aconteceu de verdade. Nesta trilha, nove em cada dez vezes
são as duas listas trocadas — **posições** em `cat_idxs`, **cardinalidades** em `cat_dims`.

> ⚠️ As células não executam enquanto houver lacunas. Preencham todas, inclusive as dos
> modelos que vocês não escolheram: o Python lê a célula inteira antes de rodar.

In [ ]:
#@title 🧰 Peças prontas: as duas redes e o laço de treino { display-mode: "form" }
# Ninguém precisa escrever estas peças. Mas, quando o treino der errado, o problema vai
# estar numa delas: leiam com calma.
import time, copy, subprocess, sys
import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(SEMENTE)
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"


class RedeDensa(nn.Module):
    """entrada → [Linear → ReLU → Dropout] × camadas → saída (uma nota por classe)."""
    def __init__(self, entradas, saidas, largura=(64,), dropout=0.0):
        super().__init__()
        camadas, n = [], entradas
        for w in largura:
            camadas += [nn.Linear(n, w), nn.ReLU(), nn.Dropout(dropout)]
            n = w
        camadas.append(nn.Linear(n, saidas))
        self.rede = nn.Sequential(*camadas)

    def forward(self, x_num, x_cat):
        return self.rede(x_num)


class RedeComEmbeddings(nn.Module):
    """Cada coluna de categoria ganha uma tabela de vetores; o resto é a rede densa."""
    def __init__(self, cardinalidades, n_numericas, saidas, dim_embedding=8,
                 largura=(64,), dropout=0.0):
        super().__init__()
        self.embeddings = nn.ModuleList(
            [nn.Embedding(c, dim_embedding) for c in cardinalidades])
        self.corpo = RedeDensa(n_numericas + dim_embedding * len(cardinalidades),
                               saidas, largura, dropout)

    def forward(self, x_num, x_cat):
        vetores = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        return self.corpo(torch.cat([x_num] + vetores, dim=1), None)


def carregador(x_num, x_cat, y, lote, embaralhar):
    t = lambda a, tipo: torch.as_tensor(np.asarray(a), dtype=tipo)
    return DataLoader(TensorDataset(t(x_num, torch.float32), t(x_cat, torch.long),
                                    t(y, torch.long)), batch_size=lote, shuffle=embaralhar)


@torch.no_grad()
def prever_prob(modelo, x_num, x_cat):
    modelo.eval()  # dropout desligado: na validação todos os neurônios trabalham
    saida = modelo(torch.as_tensor(np.asarray(x_num), dtype=torch.float32, device=DISPOSITIVO),
                   torch.as_tensor(np.asarray(x_cat), dtype=torch.long, device=DISPOSITIVO))
    return torch.softmax(saida, dim=1).cpu().numpy()


def treinar(modelo, dados_tr, dados_va, otimizador, agendador, perda,
            epocas, paciencia, ponto_de_verificacao):
    """O laço de treino. Não é para escrever: é para entender a ordem."""
    historico = {"treino": [], "validacao": []}
    melhor_perda, melhor_epoca, melhor_estado, sem_melhorar = float("inf"), 0, None, 0
    for epoca in range(1, epocas + 1):
        modelo.train()
        soma, n = 0.0, 0
        # uma volta por lote: cada lote gera UM ajuste dos pesos
        for x_num, x_cat, resposta in dados_tr:
            x_num, x_cat, resposta = (x_num.to(DISPOSITIVO), x_cat.to(DISPOSITIVO),
                                      resposta.to(DISPOSITIVO))
            otimizador.zero_grad()            # 1 · apaga o passo anterior
            saida = modelo(x_num, x_cat)      # 2 · passagem para a frente
            erro = perda(saida, resposta)     # 3 · o quanto errou
            erro.backward()                   # 4 · propaga o erro para trás
            otimizador.step()                 # 5 · anda (só aqui o modelo muda)
            soma += erro.item() * len(resposta); n += len(resposta)
        if agendador is not None:
            agendador.step()                  # o passo encurta a cada época
        historico["treino"].append(soma / n)

        modelo.eval()
        with torch.no_grad():
            s, m = 0.0, 0
            for x_num, x_cat, resposta in dados_va:
                x_num, x_cat, resposta = (x_num.to(DISPOSITIVO), x_cat.to(DISPOSITIVO),
                                          resposta.to(DISPOSITIVO))
                s += perda(modelo(x_num, x_cat), resposta).item() * len(resposta)
                m += len(resposta)
        historico["validacao"].append(s / m)

        if historico["validacao"][-1] < melhor_perda:
            melhor_perda, melhor_epoca, sem_melhorar = historico["validacao"][-1], epoca, 0
            if ponto_de_verificacao:          # guarda a MELHOR época, não a última
                melhor_estado = copy.deepcopy(modelo.state_dict())
        else:
            sem_melhorar += 1
            if sem_melhorar >= paciencia:     # parada antecipada
                break
    if ponto_de_verificacao and melhor_estado is not None:
        modelo.load_state_dict(melhor_estado)
    historico["melhor_epoca"] = melhor_epoca
    historico["ultima_epoca"] = len(historico["treino"])
    return historico


def instalar_tabnet():
    try:
        import pytorch_tabnet  # noqa: F401
    except ImportError:
        print("Instalando pytorch-tabnet (só na primeira vez)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-tabnet"], check=True)


print(f"Peças prontas. Dispositivo: {DISPOSITIVO}.")

In [ ]:
# ═══ CÉLULA A · preparo ═══
# Substituam cada ____ pelo que falta.
X = dados.drop(columns=[ALVO])
y = dados[ALVO].to_numpy()

# LACUNA 3 ▸ separar mantendo a proporção das classes: que vetor vai no stratify?
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=tamanho_da_validacao/100, random_state=SEMENTE, stratify=____)

# Categoria vira índice inteiro, sempre com o vocabulário do conjunto PÚBLICO.
# (Saber quais valores existem não é vazamento: não usa resposta nenhuma.)
VOCABULARIO = {c: {v: i for i, v in enumerate(sorted(dados[c].astype(str).unique()))}
               for c in CATEGORICAS}
codificar = lambda df: np.stack([df[c].astype(str).map(VOCABULARIO[c]).to_numpy()
                                 for c in CATEGORICAS], axis=1)

if modelo_escolhido == "Rede densa":
    # categoria vira coluna de 0 e 1: o número da categoria não tem ordem
    # LACUNA 4 ▸ quais colunas se abrem em 0 e 1?
    abrir = lambda df: pd.get_dummies(df.astype({c: str for c in CATEGORICAS}),
                                      columns=____, dtype=float)
    X_tr_p = abrir(X_tr)
    X_va_p = abrir(X_va).reindex(columns=X_tr_p.columns, fill_value=0)
    X_te_p = abrir(dados_teste).reindex(columns=X_tr_p.columns, fill_value=0)
    # LACUNA 5 ▸ a régua aprende média e desvio SÓ com qual conjunto?
    regua = StandardScaler().fit(____)
    X_tr_num, X_va_num, X_te_num = (regua.transform(d) for d in (X_tr_p, X_va_p, X_te_p))
    X_tr_cat, X_va_cat, X_te_cat = (np.zeros((len(d), 0)) for d in (X_tr, X_va, dados_teste))

elif modelo_escolhido == "Rede com embeddings":
    # LACUNA 6 ▸ a régua vale SÓ para as colunas numéricas, e SÓ do treino.
    regua = StandardScaler().fit(____)
    X_tr_num, X_va_num, X_te_num = (regua.transform(d[NUMERICAS])
                                    for d in (X_tr, X_va, dados_teste))
    X_tr_cat, X_va_cat, X_te_cat = (codificar(d) for d in (X_tr, X_va, dados_teste))

else:  # TabNet: categorias como índices, sem régua (ele normaliza por dentro)
    ORDEM = NUMERICAS + CATEGORICAS
    juntar = lambda d: np.hstack([d[NUMERICAS].to_numpy(dtype=float), codificar(d)])
    X_tr_t, X_va_t, X_te_t = (juntar(d) for d in (X_tr, X_va, dados_teste))
    # onde cada coluna de categoria está na tabela, e quantos valores cada uma tem
    idx_cat = [ORDEM.index(c) for c in CATEGORICAS]
    dim_cat = [cardinalidades[c] for c in CATEGORICAS]

print(f"Treino: {len(y_tr)}  |  Validação: {len(y_va)}  |  Modelo: {modelo_escolhido}")
if modelo_escolhido == "Rede densa":
    print(f"Entradas da rede densa: {X_tr_num.shape[1]} colunas (categorias abertas em 0 e 1)")
elif modelo_escolhido == "Rede com embeddings":
    print(f"Numéricas: {X_tr_num.shape[1]}  |  categorias como índices: {X_tr_cat.shape[1]}")
else:
    print(f"idx_cat = {idx_cat}\ndim_cat = {dim_cat}")

In [ ]:
# ═══ CÉLULA B · treino ═══
# ═══ ZONA LIVRE ═══  (vazia = resultado correto e mediano; um ajuste por vez)
peso_por_classe = None        # None ou "balanceado"
decaimento_de_pesos = 0.01    # weight_decay do AdamW
# ══════════════════
ponto_de_verificacao = ____   # LACUNA 7 ▸ guardar a melhor época? (True ou False)
inicio = time.time()

if modelo_escolhido == "TabNet":
    instalar_tabnet()
    from pytorch_tabnet.tab_model import TabNetClassifier
    modelo = TabNetClassifier(
        # LACUNAS 8 e 9 ▸ as posições das categorias e quantos valores cada uma tem.
        #                 Trocar uma pela outra é o engano mais comum desta trilha.
        cat_idxs=____, cat_dims=____, cat_emb_dim=2,
        n_d=n_d, n_a=n_d, n_steps=n_steps, seed=SEMENTE, verbose=0,
        # LACUNA 10 ▸ a taxa de aprendizado escolhida no Bloco 2
        optimizer_fn=torch.optim.AdamW, optimizer_params=dict(lr=____),
        scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
        scheduler_params=dict(T_max=epocas), device_name=DISPOSITIVO)
    # LACUNA 11 ▸ o lote virtual precisa DIVIDIR o lote exatamente (lote = {lote})
    lote_virtual = ____
    modelo.fit(X_tr_t, y_tr, eval_set=[(X_va_t, y_va)], eval_name=["validacao"],
               eval_metric=["logloss"], max_epochs=epocas, patience=paciencia,
               batch_size=lote, virtual_batch_size=lote_virtual,
               weights=1 if peso_por_classe == "balanceado" else 0)
    # O TabNet guarda a melhor época sozinho quando há paciência.
    historico = {"treino": list(modelo.history["loss"]),
                 "validacao": list(modelo.history["validacao_logloss"]),
                 "melhor_epoca": modelo.best_epoch + 1}
    historico["ultima_epoca"] = len(historico["treino"])

else:
    if modelo_escolhido == "Rede densa":
        modelo = RedeDensa(X_tr_num.shape[1], len(NOMES), largura=camadas, dropout=dropout)
    else:
        modelo = RedeComEmbeddings([cardinalidades[c] for c in CATEGORICAS],
                                   X_tr_num.shape[1], len(NOMES), dim_embedding=dim_embedding,
                                   largura=camadas, dropout=dropout)
    modelo = modelo.to(DISPOSITIVO)
    # LACUNA 12 ▸ quem move os pesos depois que o erro apontou a direção?
    otimizador = ____(modelo.parameters(), lr=taxa, weight_decay=decaimento_de_pesos)
    # LACUNA 13 ▸ o passo vai encurtando ao longo do treino (agendador cosseno)
    agendador = ____(otimizador, T_max=epocas)
    pesos = None
    if peso_por_classe == "balanceado":
        pesos = torch.tensor(len(y_tr) / (len(NOMES) * np.bincount(y_tr, minlength=len(NOMES))),
                             dtype=torch.float32, device=DISPOSITIVO)
    perda = nn.CrossEntropyLoss(weight=pesos)
    historico = treinar(modelo,
                        carregador(X_tr_num, X_tr_cat, y_tr, lote, embaralhar=True),
                        carregador(X_va_num, X_va_cat, y_va, 1024, embaralhar=False),
                        otimizador, agendador, perda, epocas, paciencia, ponto_de_verificacao)

tempo_de_treino = time.time() - inicio
print(f"Treino concluído em {tempo_de_treino:.0f} s · {historico['ultima_epoca']} épocas · "
      f"melhor época: {historico['melhor_epoca']}")

In [ ]:
#@title 🔎 Verificação das lacunas (inclusive o erro que não dá erro) { display-mode: "form" }
erros = []
try:
    if not np.isclose(piso, f1_score(y_publico, chute, average="macro")):
        erros.append("LACUNA 1: o piso não é o F1-macro do chute preguiçoso.")
except NameError:
    erros.append("LACUNA 1: a variável 'piso' não existe. Rodem o Bloco 1.")
try:
    if cardinalidades != {c: int(dados[c].nunique()) for c in CATEGORICAS}:
        erros.append("LACUNA 2: as cardinalidades não são o número de valores diferentes.")
except NameError:
    erros.append("LACUNA 2: 'cardinalidades' não existe. Rodem o Bloco 1.")
try:
    _esp = np.bincount(y, minlength=len(NOMES)) * len(y_tr) / len(y)
    if np.abs(np.bincount(y_tr, minlength=len(NOMES)) - _esp).max() > 1.5:
        erros.append("LACUNA 3: o treino não manteve a proporção das classes (stratify).")
except NameError:
    erros.append("LACUNA 3: X_tr/y_tr não existem. Rodem a célula A.")
if modelo_escolhido == "Rede densa":
    if any(c in X_tr_p.columns for c in CATEGORICAS):
        erros.append("LACUNA 4: as colunas de categoria não foram abertas em 0 e 1.")
    if regua.n_samples_seen_ != len(y_tr):
        erros.append(f"LACUNA 5: VAZAMENTO. A régua viu {int(regua.n_samples_seen_)} linhas; o treino "
                     f"tem {len(y_tr)}. Ela aprendeu com dados de fora do treino, e a nota "
                     "de validação fica melhor do que a da plataforma.")
elif modelo_escolhido == "Rede com embeddings":
    if regua.n_samples_seen_ != len(y_tr) or regua.n_features_in_ != len(NUMERICAS):
        erros.append("LACUNA 6: a régua precisa ver SÓ as numéricas, SÓ do treino "
                     f"(viu {int(regua.n_samples_seen_)} linhas e {regua.n_features_in_} colunas).")
else:
    if list(modelo.cat_idxs) != idx_cat or list(modelo.cat_dims) != dim_cat:
        erros.append("LACUNAS 8 e 9: cat_idxs recebe as POSIÇÕES e cat_dims as CARDINALIDADES.")
    if lote % lote_virtual:
        erros.append(f"LACUNA 11: o lote virtual ({lote_virtual}) não divide o lote ({lote}).")
    if not np.isclose(modelo.optimizer_params["lr"], taxa):
        erros.append("LACUNA 10: o TabNet não recebeu a taxa escolhida no Bloco 2.")
if ponto_de_verificacao is not True:
    erros.append("LACUNA 7: sem ponto de verificação, a entrega é o modelo da última época.")
if modelo_escolhido != "TabNet":
    if type(otimizador).__name__ != "AdamW":
        erros.append(f"LACUNA 12: esperado AdamW, obtido {type(otimizador).__name__}.")
    if type(agendador).__name__ != "CosineAnnealingLR":
        erros.append(f"LACUNA 13: esperado CosineAnnealingLR, obtido {type(agendador).__name__}.")

if erros:
    print("VERIFICAÇÃO REPROVADA. Pontos a revisar:\n")
    for e in erros: print("   [!] " + e + "\n")
    # O caderno para aqui: seguir com uma lacuna errada é deixar a nota mentir.
    raise RuntimeError("VERIFICAÇÃO REPROVADA: corrijam os pontos acima antes de seguir.")
else:
    print("VERIFICAÇÃO APROVADA\n")
    print("   Piso e cardinalidades certos, régua só no treino, peças configuradas.")
    if peso_por_classe is None:
        print("\n   A Zona Livre está como veio. Correto e mediano: o que vocês acrescentarem")
        print("   ali (um ajuste por vez) é o que separa o mediano do bom.")

In [ ]:
# ═══ CÉLULA C · avaliação ═══  (pronta: não é para escrever, é para ler)
if modelo_escolhido == "TabNet":
    prob_va = modelo.predict_proba(X_va_t)
else:
    prob_va = prever_prob(modelo, X_va_num, X_va_cat)
previsoes = prob_va.argmax(axis=1)
f1 = f1_score(y_va, previsoes, average="macro")
acuracia = accuracy_score(y_va, previsoes)
recall_critica = recall_score(y_va, previsoes, labels=[2], average="macro")

if "registro" not in dir():
    registro = []
registro.append({"modelo": modelo_escolhido, "taxa": taxa, "camadas": str(camadas),
                 "dropout": dropout, "peso_por_classe": peso_por_classe,
                 "melhor_epoca": historico["melhor_epoca"], "f1_macro": round(f1, 4),
                 "minutos": round(tempo_de_treino / 60, 2)})

print("="*60)
print("            DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("="*60)
print(f"  Piso (chute preguiçoso) ............... {piso*100:5.1f}%")
print(f"  ⭐ F1-MACRO (métrica oficial) .......... {f1*100:5.1f}%")
print(f"     Acurácia ............................ {acuracia*100:5.1f}%")
print(f"     Revocação de '{NOMES[2]}' (apoio) ....... {recall_critica*100:5.1f}%")
print("="*60)
a, s_, i_ = (0.7, 0.66, 0.6)
if   f1 >= a:  print("  Desempenho alto. Confiram a forma das curvas antes de comemorar.")
elif f1 >= s_: print("  Desempenho sólido. Há margem na Zona Livre e nos botões do Bloco 2.")
elif f1 >= i_: print("  Desempenho intermediário. Olhem as curvas: o formato diz qual botão mexer.")
else:          print("  Desempenho baixo. Comparem com o piso e leiam as curvas antes de mudar algo.")

---
# 📈 BLOCO 4 · AVALIAR
## *Na Fase 1 vocês tinham um número no fim. Agora vocês têm um filme.*

Quatro formatos de curva, quatro diagnósticos:

| Formato | Diagnóstico | O que fazer |
|:---|:---|:---|
| As duas descem e ficam juntas | **Treino saudável** | Nada: guardem a melhor época |
| Treino despenca, validação sobe | **Sobreajuste** | Parada antecipada, mais dropout, rede menor |
| As duas param altas e juntas | **Subajuste** | Mais capacidade, mais taxa, mais épocas; conferir a régua |
| Serrilhada e subindo | **Taxa alta demais** | Dividir a taxa por dez — sempre a primeira coisa a tentar |

In [ ]:
#@title 📉 As duas curvas e o diagnóstico { display-mode: "form" }
tr, va = np.array(historico["treino"]), np.array(historico["validacao"])
ep = np.arange(1, len(tr) + 1)
melhor = historico["melhor_epoca"]
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(ep, tr, "-", color="#3B5BDB", lw=2, label="treino")
ax.plot(ep, va, "--", color="#E6177A", lw=2, label="validação")
ax.axvline(melhor, color="#2F9E44", ls=":", lw=2, label=f"melhor época ({melhor})")
ax.set_xlabel("épocas →"); ax.set_ylabel("perda (erro)"); ax.legend()
ax.set_title("Perda de treino e de validação", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

_subidas = np.mean(np.diff(va) > 0) if len(va) > 2 else 0
if len(va) > 3 and _subidas > .45 and va[-1] > va.min() * 1.05 and tr[-1] > tr.min() * 1.02:
    diag = "TAXA ALTA DEMAIS: a validação serrilha. Dividam a taxa por dez."
elif va[-1] > va.min() * 1.08 and tr[-1] < tr[melhor - 1]:
    diag = "SOBREAJUSTE: treino caindo, validação subindo depois da melhor época."
elif melhor >= len(va) - 1 and historico["ultima_epoca"] >= epocas:
    diag = ("SUBAJUSTE OU TAXA BAIXA: a melhor época é a última e as épocas acabaram com a "
            "curva ainda descendo. Aumentem a taxa (ou as épocas) antes de mexer na rede.")
else:
    diag = "TREINO SAUDÁVEL: as duas descem e a distância se estabiliza."
print(f"DIAGNÓSTICO: {diag}")
print(f"   distância entre as curvas na melhor época: {va[melhor-1] - tr[melhor-1]:+.3f}")
print(f"   a entrega leva a época {melhor}, não a {len(va)}." if melhor != len(va)
      else "   a melhor época foi a última.")

In [ ]:
#@title 🧩 Onde o erro mora: a matriz em contagem e em proporção { display-mode: "form" }
cm = confusion_matrix(y_va, previsoes, labels=list(NOMES))
prop = cm / cm.sum(axis=1, keepdims=True)
rot = list(NOMES.values())
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
for ax, m, titulo, fmt in [(axes[0], cm, "em contagem", "{:d}"),
                           (axes[1], prop, "em proporção da linha", "{:.0%}")]:
    ax.imshow(m, cmap="Purples")
    for i in range(len(rot)):
        for j in range(len(rot)):
            ax.text(j, i, fmt.format(m[i, j]), ha="center", va="center",
                    color="white" if m[i, j] > m.max() * .6 else "black", fontweight="bold")
    ax.set_xticks(range(len(rot))); ax.set_xticklabels(rot)
    ax.set_yticks(range(len(rot))); ax.set_yticklabels(rot)
    ax.set_xlabel("o que o modelo respondeu"); ax.set_ylabel("o que o caso era")
    ax.set_title(titulo, fontweight="bold")
plt.tight_layout(); plt.show()
print("Leiam em proporção: 40 erros numa classe grande podem ser menos graves que 12 numa")
print("classe pequena. E perguntem: os pares confundidos são vizinhos (médio × alto) ou")
print("extremos (baixo × alto)? Errar entre vizinhos é diferente de errar entre distantes.\n")
print(classification_report(y_va, previsoes, labels=list(NOMES), target_names=rot, digits=3))

In [ ]:
#@title 🌲 A régua da Fase 1: a floresta no mesmo recorte { display-mode: "form" }
# A rede só se justifica se superar o que a Fase 1 já entregava, no MESMO treino/validação.
from sklearn.ensemble import RandomForestClassifier
_ab = lambda d: pd.get_dummies(d.astype({c: str for c in CATEGORICAS}), columns=CATEGORICAS, dtype=float)
_Xtr = _ab(X_tr); _Xva = _ab(X_va).reindex(columns=_Xtr.columns, fill_value=0)
_t = time.time()
_rf = RandomForestClassifier(n_estimators=400, max_depth=10, min_samples_leaf=3,
                             class_weight="balanced", random_state=SEMENTE, n_jobs=-1).fit(_Xtr, y_tr)
f1_fase1 = f1_score(y_va, _rf.predict(_Xva), average="macro")
print(f"   Floresta (Fase 1) ....... F1-macro {f1_fase1*100:5.1f}%   em {time.time()-_t:4.1f} s")
print(f"   {modelo_escolhido:<24} F1-macro {f1*100:5.1f}%   em {tempo_de_treino:4.1f} s")
print(f"   Piso .................... F1-macro {piso*100:5.1f}%")
if f1 < f1_fase1:
    print("\nA rede ainda não supera a floresta. Isso é um resultado — e vira um parágrafo")
    print("honesto no relatório se vocês não conseguirem virar o jogo na Zona Livre.")

In [ ]:
#@title ⏱️ Custo contra ganho: todos os experimentos desta sessão { display-mode: "form" }
tab = pd.DataFrame(registro)
display(tab)
if len(tab) > 1:
    fig, ax = plt.subplots(figsize=(7.5, 4))
    for m, cor in zip(["Rede densa", "Rede com embeddings", "TabNet"], ["#3B5BDB", "#2F9E44", "#E6177A"]):
        s = tab[tab.modelo == m]
        ax.scatter(s.minutos, s.f1_macro, s=70, color=cor, label=m)
    ax.axhline(f1_fase1, color="#999", ls="--", lw=1, label="floresta (Fase 1)")
    ax.set_xlabel("minutos de treino"); ax.set_ylabel("F1-macro (validação)"); ax.legend()
    ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
print("A pergunta não é qual acerta mais: é quanto cada ponto a mais custou. Um centésimo")
print("por cinco vezes o tempo só se justifica se o relatório usar o que o modelo mais caro")
print("entrega a mais (no TabNet, a explicação de quais colunas ele olhou).")

In [ ]:
#@title 👥 A média engana: F1-macro por grupo { display-mode: "form" }
grupo = "turno"  #@param ["turno", "genero", "bolsista", "deslocado", "curso"]
_g = X_va[grupo].to_numpy()
linhas = []
for v in sorted(pd.unique(_g), key=str):
    m = _g == v
    if m.sum() >= 15:
        linhas.append({"grupo": f"{grupo} = {v}", "casos": int(m.sum()),
                       "F1-macro": round(f1_score(y_va[m], previsoes[m], average="macro"), 3)})
tab_g = pd.DataFrame(linhas)
display(tab_g)
if len(tab_g) > 1:
    dif = tab_g["F1-macro"].max() - tab_g["F1-macro"].min()
    print(f"Diferença entre o melhor e o pior grupo: {dif*100:.1f} pontos.")
    if dif > .08:
        print("A média geral esconde um grupo que vai pior. Isso vai para o relatório.")

In [ ]:
#@title 🔦 Quais colunas o TabNet olhou? { display-mode: "form" }
if modelo_escolhido == "TabNet":
    imp = pd.Series(modelo.feature_importances_, index=ORDEM).sort_values().tail(15)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(imp.index, imp.values, color="#E6177A")
    ax.set_title("Importância das colunas segundo o TabNet", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
    print("É a única das três que entrega isto de graça. Comparem com o medidor de sinal")
    print("do Bloco 1. Lembrem: importância não é causa.")
else:
    print("Esta célula só funciona com o TabNet: as redes densas não dizem quais colunas usaram.")
    print("Se o relatório de vocês precisa dessa explicação, esse é o argumento para pagar o")
    print("tempo extra do TabNet.")

---
# 🤔 PARADA OBRIGATÓRIA

> **Marina:** *"A rede não me diz por que decidiu. A floresta também não dizia muito, mas
> pelo menos eu conseguia desenhar uma árvore no quadro."*

1. **Caixa-preta.** Das três arquiteturas, só o TabNet diz quais colunas usou. Se a escola
   precisar explicar a uma família por que o filho apareceu na lista, qual modelo vocês
   entregariam, mesmo que ele não seja o de maior F1?
2. **A média engana.** A célula de grupos mostrou o F1 por turno, gênero, bolsa. Se um grupo
   vai bem pior, o sistema trata esse grupo de forma diferente. O que fazer com isso?
3. **Variáveis socioeconômicas.** `devedor`, `mensalidades_em_dia`, `bolsista` e a ocupação
   dos pais pesam. Removê-las resolve ou só esconde, já que outras colunas carregam a
   mesma informação?
4. **IA generativa.** Se usaram, registrem qual, para quê, como conferiram e o que mudaram.

In [ ]:
#@title 📝 O relatório da sua equipe { display-mode: "form" }
#@markdown Vale nota e decide o desempate: números e observações concretas.
nome_da_equipe = ""  #@param {type:"string"}
#@markdown **1. Piso, três colunas de maior sinal e cardinalidades: o que o Bloco 1 mostrou?**
resposta_1 = ""  #@param {type:"string"}
#@markdown **2. Tabela de experimentos: o que cada ajuste fez no F1 e na forma das curvas?**
resposta_2 = ""  #@param {type:"string"}
#@markdown **3. A rede superou a floresta da Fase 1? Valeu o custo? Qual grupo vai pior?**
resposta_3 = ""  #@param {type:"string"}
#@markdown **4. Que cuidado com variáveis socioeconômicas e com o uso da lista vocês anexariam?**
resposta_4 = ""  #@param {type:"string"}
#@markdown **5. Usaram IA generativa? Qual, para quê, como conferiram e o que mudaram?**
resposta_5 = ""  #@param {type:"string"}

relatorio = {"equipe": nome_da_equipe, "modelo": modelo_escolhido, "taxa": taxa,
             "epocas": epocas, "melhor_epoca": historico["melhor_epoca"], "lote": lote,
             "camadas": str(camadas), "dropout": dropout, "peso_por_classe": peso_por_classe,
             "validacao_pct": tamanho_da_validacao, "piso": round(piso, 4),
             "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
             "f1_floresta_fase1": round(f1_fase1, 4),
             "just_exploracao": resposta_1, "just_experimentos": resposta_2,
             "just_custo_e_grupos": resposta_3, "just_etica": resposta_4,
             "uso_ia_generativa": resposta_5}
faltando = [k for k in ["equipe", "just_exploracao", "just_experimentos",
                        "just_custo_e_grupos", "just_etica", "uso_ia_generativa"]
            if not str(relatorio[k]).strip()]
print("⚠️ Ainda faltam campos: " + ", ".join(faltando) if faltando
      else "Relatório completo. Prossigam para a submissão.")

In [ ]:
#@title 🚀 Gerar o arquivo de submissão { display-mode: "form" }
import json, pathlib
PASTA_SUBMISSAO = pathlib.Path("submissoes")
PASTA_SUBMISSAO.mkdir(parents=True, exist_ok=True)
if "relatorio" not in dir() or not isinstance(relatorio, dict):
    relatorio = {"equipe": "", "modelo": modelo_escolhido, "f1_macro": round(f1, 4)}

# O teste passa pelo MESMO preparo da célula A (a régua do treino, o vocabulário do público).
if modelo_escolhido == "TabNet":
    palpites = modelo.predict(X_te_t)
else:
    palpites = prever_prob(modelo, X_te_num, X_te_cat).argmax(axis=1)
submissao = pd.DataFrame({"id_estudante": IDS_TESTE, "risco_previsto": palpites})

assert len(submissao) == N_TESTE, f"A submissão precisa ter {N_TESTE} linhas."
csv_saida = PASTA_SUBMISSAO / "submissao_farol.csv"
json_saida = PASTA_SUBMISSAO / "relatorio_farol.json"
submissao.to_csv(csv_saida, index=False)
with open(json_saida, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print("="*60)
print("             ARQUIVOS PRONTOS PARA ENVIO")
print("="*60)
print(f"  submissao_farol.csv   ({len(submissao)} linhas)")
print("  relatorio_farol.json")
print("="*60)
print("\n  Distribuição prevista no teste:")
for k, v in pd.Series(palpites).value_counts().sort_index().items():
    print(f"     {NOMES[k]:<11} {v:>4}  ({v/len(palpites)*100:.1f}%)")
display(submissao.head())
print("\n  Para corrigir (da raiz do repositório):")
print(f"     python simulados/corrigir.py farol {csv_saida}")

try:
    from google.colab import files
    files.download(str(csv_saida))
    files.download(str(json_saida))
    print("\n  Download iniciado.")
except ImportError:
    print(f"\n  Fora do Colab: os arquivos estão em {PASTA_SUBMISSAO.resolve()}")

---
# 🌟 FIM DA OPERAÇÃO FAROL · FASE 2

### Competências praticadas

- Piso, sinal das colunas e cardinalidades **antes** de treinar
- Três preparos para o mesmo dado: 0/1, índices + cardinalidades, posições + cardinalidades
- Régua só no treino: o vazamento que não dá erro
- Taxa de aprendizado, largura, dropout e a leitura das duas curvas
- Otimizador, agendador, parada antecipada e ponto de verificação
- Custo contra ganho e comparação honesta com a floresta da Fase 1
- Desempenho por grupo: a média que esconde quem vai mal